# Lab 60: The Logic Gate

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-60.ipynb)

**What you will do:** score your own Wason card results against the published abstract-versus-social accuracy gap, pool the class's data, and test whether a small language model solves an abstract logic rule and a social-contract rule equally well.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 60 you tried the **Wason selection task**: given four cards (A, K, 4, 7) and the rule "if
a card has a vowel on one side, it has an even number on the other," you had to say which cards
to turn over to test the rule. The correct answer is **A and 7**, but most people choose A and
4. Across decades of replications, only a minority solve the abstract version — commonly well
under a quarter, often closer to 10% (Wason, 1968). Rephrased as a social rule ("if drinking
alcohol, must be over 21"), nearly everyone gets it right instantly, and formal studies put
accuracy at roughly 65-80% correct (Cosmides, 1989).

## Record your results

Test a few friends on the abstract card version (A, K, 4, 7) and, separately, on the bouncer
version. Record whether each person answered correctly (both target cards, no others).

In [ ]:
results = pd.DataFrame({
    "participant": ["P1", "P2", "P3", "P4"],
    "version": ["abstract", "abstract", "social", "social"],
    "correct": [False, False, True, True],  # example data: replace with your own results
})
display(results)
results.groupby("version").correct.mean().plot.bar(color=["grey", "steelblue"], figsize=(4.5, 3))
plt.ylabel("proportion correct"); plt.ylim(0, 1); plt.xticks(rotation=0); plt.show()

## Compare

The lab reports abstract-version accuracy of well under 25% (often closer to 10%), against
roughly 65-80% for the socially framed version (Cosmides, 1989).

In [ ]:
rates = results.groupby("version").correct.mean()
print(f"Your abstract accuracy: {rates.get('abstract', float('nan')):.0%}  |  published: often ~10%, well under 25%")
print(f"Your social accuracy:   {rates.get('social', float('nan')):.0%}  |  published: roughly 65-80%")
if rates.get("social", 0) > rates.get("abstract", 0):
    print("Your data shows the same direction as the published gap: content changes correctness")
    print("even though the underlying logic (modus tollens) is identical in both versions.")
else:
    print("Your small sample did not show the predicted gap. With only a few participants this")
    print("can happen by chance -- the effect is large and robust across the published literature.")

## The AI side

The book's own AI extension asks whether a chatbot solves the abstract puzzle and an altered
rule using genuine logic, or by recognising a familiar problem type. Below, a small open
instruction-following model (flan-t5-small) is tested on three versions: the original abstract
rule, the social bouncer version, and an altered abstract rule the book proposes ("if a card
shows a prime number on one side, it must show a consonant on the other" -- cards 3, 8, K, A).

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")

tasks = {
    "abstract (vowel/even)": (
        "Four cards show A, K, 4, and 7. Each card has a letter on one side and a number on "
        "the other. Rule: if a card has a vowel on one side, it has an even number on the "
        "other side. Which two cards must you turn over to test the rule? Answer with exactly "
        "two of: A, K, 4, 7."),
    "social (bouncer)": (
        "You are a bouncer. The rule is: if a person is drinking alcohol, they must be over "
        "21. You see: a beer drinker, a cola drinker, a 25-year-old, a 17-year-old. Which two "
        "must you check? Answer with exactly two of: beer drinker, cola drinker, 25-year-old, "
        "17-year-old."),
    "altered abstract (prime/consonant)": (
        "Four cards show 3, 8, K, and A. Each card has a number on one side and a letter on "
        "the other. Rule: if a card has a prime number on one side, it has a consonant on the "
        "other side. Which two cards must you turn over to test the rule? Answer with exactly "
        "two of: 3, 8, K, A."),
}
correct = {"abstract (vowel/even)": {"A", "7"},
           "social (bouncer)": {"beer drinker", "17-year-old"},
           "altered abstract (prime/consonant)": {"3", "A"}}

for label, prompt in tasks.items():
    answer = qa(prompt, max_new_tokens=40)[0]["generated_text"]
    hits = sum(1 for card in correct[label] if card.lower() in answer.lower())
    print(f"{label}\n  model answer: {answer}\n  correct cards named: {hits}/2\n")

A model that names the right two cards on the familiar abstract rule and the familiar social
rule, but fails on the altered abstract rule (same logical form, unfamiliar content), is
behaving like a pattern-matcher rather than a logician -- much like the argument the book
makes about human cheater-detection versus abstract reasoning. flan-t5-small is a very small
model (about 80 million parameters) and can fail or give incomplete answers on any of these;
treat this as an illustration of the *kind* of failure to look for, not a reliable verdict on
what language models can or cannot do -- larger, more capable models handle these tasks very
differently.

## Pool the class data

Pool percent-correct scores on the abstract and social versions from the class (anonymous IDs,
not names).

In [ ]:
import io
csv_text = """id,version,correct
P01,abstract,0
P02,abstract,0
P03,abstract,1
P04,social,1
P05,social,1
P06,social,0
"""  # example data: replace with your class CSV, e.g. pd.read_csv("wason_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
summary = df.groupby("version").correct.agg(["mean", "count"])
display(summary)
boot = {v: [df.loc[df.version == v].correct.sample(frac=1, replace=True).mean() for _ in range(3000)]
        for v in df.version.unique()}
for v, b in boot.items():
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f"{v}: pooled accuracy {summary.loc[v, 'mean']:.0%} (95% bootstrap interval {lo:.0%}-{hi:.0%}, n={int(summary.loc[v, 'count'])})")
print("Published values for comparison: abstract often ~10% (well under 25%); social ~65-80%.")

## Questions to think about

1. Did any of your friends solve the abstract card version correctly on their first try? If not, does that match the book's claim that "only a minority" — commonly well under a quarter — succeed?
2. Why does turning over the 4-card add no information about the rule "if vowel, then even," even if it does have a vowel on the back?
3. In the AI demo, did flan-t5-small solve the familiar abstract and social versions but stumble on the altered abstract rule? What would that pattern suggest about how it is solving the task?
4. The book argues that human logic is tuned for detecting cheaters, not evaluating abstract rules. Can you think of a real-world abstract rule (unrelated to social contracts) that people you know find hard to reason about correctly?

## Challenge

Write your own altered abstract rule (different content, same logical form as "if P then Q")
and test it on a friend who has already seen the vowel/even version. Does familiarity with the
logical form transfer, or does each new abstract rule feel just as hard?

In [ ]:
# Example data: replace with your own altered-rule results.
my_rule = "If a card shows a triangle, it is coloured blue."
cards = ["triangle", "square", "blue", "red"]
friend_answer = ["triangle", "red"]      # example: what your friend said to flip
correct_answer = {"triangle", "red"}     # the logically correct pair for this rule

hits = len(set(friend_answer) & correct_answer)
print(f"Rule tested: {my_rule}")
print(f"Friend named: {friend_answer} -> {hits}/2 correct")
print("Replace my_rule, cards, and friend_answer with your own altered rule and result.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-60.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")